In [3]:
#!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 7.2 MB/s eta 0:00:00


In [4]:
import torch
import torchvision.transforms as T

from ultralytics import YOLO
from ultralytics.data.dataset import ClassificationDataset
from ultralytics.models.yolo.classify import(
    ClassificationTrainer,
    ClassificationValidator
)

class CustomizedDataset(ClassificationDataset):
    def __init__(self,root:str,args,augment:bool=False,prefix: str=""):
        super().__init__(root,args,augment,prefix)
        train_transforms = T.Compose([
            T.Resize((args.imgsz,args.imgsz)),
            T.RandomHorizontalFlip(p=args.fliplr),
            T.RandomVerticalFlip(p=args.flipud),
            T.RandAugment(interpolation=T.InterpolationMode.BILINEAR),
            T.ColorJitter(
                brightness=args.hsv_v,
                contrast=args.hsv_s,
                saturation=args.hsv_s,
                hue=args.hsv_h
            ),
            T.ToTensor(),
            T.Normalize(mean=torch.tensor(0),std=torch.tensor(1)),
            T.RandomErasing(p=args.erasing,inplace=True)
        ])
        val_transforms = T.Compose([
            T.Resize((args.imgsz,args.imgsz)),
            T.ToTensor(),
            T.Normalize(mean=torch.tensor(0),std=torch.tensor(1))

            ])

        self.torch_transforms=train_transforms if augment else val_transforms

class CustomizedTrainer(ClassificationTrainer):
  def build_dataset(self,img_path:str,mode: str="train",batch=None):
    return CustomizedDataset(root =img_path,args=self.args,augment=mode == "train",prefix=mode)


class CustomizedValidator(ClassificationValidator):
  def build_dataset(self,img_path:str,mode: str="train"):
    return CustomizedDataset(root =img_path,args=self.args,augment=mode == "train",prefix=self.args.split)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
import gdown
# google drive file ID
file_id = "1TCU1nqgIe1R_dW6LTkRxlufHlCCazyJl"

# download destinationfilename
output="myfile.zip"


# Download the file
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1TCU1nqgIe1R_dW6LTkRxlufHlCCazyJl
From (redirected): https://drive.google.com/uc?id=1TCU1nqgIe1R_dW6LTkRxlufHlCCazyJl&confirm=t&uuid=0f30558f-66c4-426b-aa88-b79d118dfdfd
To: /content/myfile.zip
100%|██████████| 63.9M/63.9M [00:01<00:00, 61.7MB/s]


'myfile.zip'

In [6]:
import zipfile

with zipfile.ZipFile("myfile.zip", "r") as zip_ref:
    zip_ref.extractall("dataset")

In [7]:
# Custom training or Transfer learning

In [8]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo11n-cls.pt")
#model.train(data="imagenet1000", trainer = CustomizedTrainer, epochs=10, imgsz=224, batch=64)
model.train(data="/content/dataset/train", trainer = CustomizedTrainer, epochs=10, imgsz=224, batch=64)

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/train, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=Non

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x789f1019ddc0>
curves: []
curves_results: []
fitness: 0.974609375
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.94921875, 'metrics/accuracy_top5': 1.0, 'fitness': 0.974609375}
save_dir: PosixPath('/content/runs/classify/train')
speed: {'preprocess': 0.06776638671879276, 'inference': 0.3925755000000475, 'loss': 9.044921878142986e-05, 'postprocess': 0.000168554687540734}
top1: 0.94921875
top5: 1.0

In [9]:
# Starting training from a pretrained *.pt model

!yolo classify train data=imagenet10 model=yolo11n-cls.pt epochs=5 imgsz=224

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=imagenet10, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize

In [10]:
metrics = model.val(data="/content/dataset/valid", validator=CustomizedValidator, imgsz=224, batch=64)

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-cls summary (fused): 47 layers, 1,528,586 parameters, 0 gradients, 3.2 GFLOPs
WARNING ⚠️ Dataset 'split=train' not found at /content/dataset/valid/train
Found 364 images in subdirectories. Attempting to split...
Splitting /content/dataset/valid (2 classes, 364 images) into 80% train, 20% val...
Split complete in /content/dataset/valid_split ✅
train: /content/dataset/valid_split/train... found 290 images in 2 classes ✅ 
val: /content/dataset/valid_split/val... found 74 images in 2 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 870.8±357.8 MB/s, size: 28.1 KB)
val: Scanning /content/dataset/valid_split/val... 74 images, 0 corrupt: 100% ━━━━━━━━━━━━ 74/74 2.8Kit/s 0.0s
val: New cache created: /content/dataset/valid_split/val.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 1.1it/s 1.8s
                   all      0.959          1
Speed: 2.2ms pre

In [13]:
metrics.top1 # top1 accuracy


0.9594594836235046

In [15]:
metrics.top5 # top5 accuracy

1.0

In [16]:
model=YOLO("/content/runs/classify/train/weights/best.pt")

#predict with the model
results=model("/content/dataset/test/daisy/1354396826_2868631432_m_jpg.rf.409eee37613d16dbc71365cb5615327e.jpg")


image 1/1 /content/dataset/test/daisy/1354396826_2868631432_m_jpg.rf.409eee37613d16dbc71365cb5615327e.jpg: 224x224 daisy 1.00, dandelion 0.00, 3.9ms
Speed: 4.4ms preprocess, 3.9ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
